# Optimizing Model Parameters
Adesso che abbiamo il nostro modello e i dati é il momento di addestrare il modello. Il training é un processo iterativo dove il modello cerca di indovinare l'output, calcola l'errore in base ai suoi parametri e li ottimizza.

Prendiamo il codice utilizzato nei precedenti notebook

In [ ]:
import torch
from networkx.algorithms.shortest_paths import weighted
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
model = NeuralNetwork().to(device)

## Hyperparameters
Gli hyperparameters sono dei parametri modificabili che ci permettono di controllare il processo di ottimizzazione del modello. Per il training definiamo i seguenti hyperparameters:
- **Number of Epochs**: Il numero di volte in cui dobbiamo iterare sul dataset
- **Batch Size**: Il numero di samples propagati nella rete prima che i parametri vengano aggiornati
- **Learning Rate**: Di quanto vanno modificati i parametri del modello ad ogni batch / epoch. Valori piccoli comportano una velocitá di apprendimento lenta ma valori grandi portano a comportamenti inaspettati.

In [ ]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

## Optimizing loop
Adesso che abbiamo impostato gli hyperparameters possiamo trainare il modello. Ogni iterazione dell'optimization loop si chiama **epoch**, ogni epoch é composta da due parti principali:
- **Train Loop**: Itera sul training dataset e prova a raggiungere dei parametri ottimali
- **Validation/Test Loop**: Itera sul test dataset per vedere se le performance del modello stanno aumentando.

Vediamo alcuni concetti chiave dell'optimization loop

### Loss Function
Ci dice l'errore che compie il modello, é la funzione che vogliamo minimizzare durante il training.
Alcune loss functions comuni includono `nn.MSELoss (Mean Square Error)` per regression e `nn.NLLLoss (Negative Log Likelihood)` per classification. `nn.CrossEntropyLoss` combina `nn.LogSoftmax` e `nn.NLLLoss`.

Passiamo i logits di output del nostro modello a `nn.CrossEntropyLoss` che li normalizza e calcola l'errore del calcolo fatto dal modello.

In [ ]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

### Optimizer
L'optimization é il processo in cui aggiustiamo i parametri del modello per ridurre l'errore ad ogni training step. Gli algoritmi di optimization definiscono come avviene questo processo. Tutta la logica dell'optimization é incapsulata nell'ogetto `optimizer`, qui useremo l'SGD Optimizer (Stochastic Gradient Descent).

Inizializziamo l'optimizer registrando i parametri del modello e il learning rate

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

All'interno del training loop, l'optimization avviene in 3 step:
- Chiama `optimizer.zero_grad()` per resettare i gradients dei parametri del modello. Di base vengono sommati ma ad ogni iterazione in questo modo li resettiamo
- Calcoliamo i gradients facendo backpropagation chiamando `loss.backward()`
- Una volta ottenuti i gradients chiamando `optimizer.step()` aggiustiamo i parametri in abse ai gradients raccolti nello step precedente

Vediamo l'implementazione completa

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Impostiamo il modello in modalitá training, serve per alcune impostazioni e best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Spostiamo i dati sulla GPU
        X, y = X.to(device), y.to(device)
        # Calcoliamo la prediction del modello e la loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    # Impostiamo il modello in evaluation mode
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Testare il modell con torch.no_grad() ci assicura che non vengano calcolati i gradients, risparmiamo anche memoria e calcoli !!
    with torch.no_grad():
        for X, y in dataloader:
            # Spostiamo i dati sulla GPU
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

Adesso possiamo chiamare i due loop, per incrementare l'apprendimento del modello possiamo anche aumentare le epochs

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 100
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

## Save and Load the Model
I modelli PyTorch salvano i loro parametri in un dizionario chiamato `state_dict` che puó essere salvato con il metodo `torch.save`

In [ ]:
torch.save(model.state_dict(), 'model_weights.pth')

Per caricare i parametri dobbiamo prima creare un'istanza del modello e poi caricarli utilizzando `load_state_dict()`

In [ ]:
model2 = NeuralNetwork().to(device)
model2.load_state_dict(torch.load('model_weights.pth', weights_only=True))
model2.eval()

### Saving and Loading Models with Shapes
Per caricare i parametri abbiamo visto che dobbiamo prima creare un modello con la stessa struttura, possiamo peró salvare l'interno modello e quindi anche la sua struttura

In [ ]:
torch.save(model, 'model.pth')

Ovviamente possiamo poi caricare l'intero modello

In [ ]:
model = torch.load('model.pth', weights_only=False)

La best practice rimane peró salvare soltanto i parametri